# 01 - Preparação de Dados

Carrega o arquivo `espelhos_acordaos_artigo2026.parquet` e o complementa com:
1. **Texto integral** de cada decisão, obtido dos ZIPs publicados na Base de Dados Abertos do STJ (dataset `integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica`).
2. **Dados do espelho** de cada acórdão (tese jurídica, tema, referências legislativas, etc.), obtidos dos JSONs de espelhos publicados no Portal de Dados Abertos do STJ.

- Dados Abertos: https://dadosabertos.web.stj.jus.br/group/jurisprudencia
- Sobre o espelho do acórdão: https://scon.stj.jus.br/docs_internet/jurisprudencia/ajuda/Espelho_do_Acordao_atualizado.pdf
- Sobre a api CKAN: https://docs.ckan.org/en/2.9/api/

No caso de haver problemas com o download de algum arquivo usando a api, pode-se baixar manualmente o arquivo e colocar na pasta de cache:
- Download manual dos arquivos com as íntegras: https://dadosabertos.web.stj.jus.br/dataset/integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica
- Download manual dos arquivos com os espelhos: https://dadosabertos.web.stj.jus.br/organization/superior-tribunal-de-justica?tags=jurisprudencia

## Pipeline
```
parquet → set(seq_documento_acordao) + set(num_registro)
         ↓
CKAN → lista de ZIPs (íntegras) + lista de JSONs (espelhos por órgão)
         ↓  (download com cache)
ZIP em memória → filtro por seq_documento → coluna 'texto'
JSON de espelho → filtro por num_registro  → colunas 'teseJuridica', 'tema', etc.
         ↓
parquet enriquecido com texto + campos do espelho
```

## 1. Configuração e Imports

In [ ]:
import os
import io
import json
import zipfile
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ─── CKAN ─────────────────────────────────────────────────────────────────────
CKAN_BASE_URL = 'https://dadosabertos.web.stj.jus.br'

# Dataset com os textos integrais das decisões (ZIPs de TXTs)
DATASET_TEXTOS = 'integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica'

# Datasets de espelhos de acórdãos por órgão julgador (sigla, dataset_id CKAN)
DATASETS_ESPELHOS = [
    ('S3', 'espelhos-de-acordaos-terceira-secao'),
    ('S2', 'espelhos-de-acordaos-segunda-secao'),
    ('S1', 'espelhos-de-acordaos-primeira-secao'),
    ('T1', 'espelhos-de-acordaos-primeira-turma'),
    ('T2', 'espelhos-de-acordaos-segunda-turma'),
    ('T3', 'espelhos-de-acordaos-terceira-turma'),
    ('T4', 'espelhos-de-acordaos-quarta-turma'),
    ('T5', 'espelhos-de-acordaos-quinta-turma'),
    ('T6', 'espelhos-de-acordaos-sexta-turma'),
    ('CE', 'espelhos-de-acordaos-corte-especial'),
]

# Campos do JSON de espelho que serão importados para o parquet
COLUNAS_ESPELHO = [
    'id',                        # identificador do espelho no CKAN
    'siglaClasse',
    'descricaoClasse',
    'nomeOrgaoJulgador',
    'ministroRelator',
    'tipoDeDecisao',
    'dataDecisao',
    'jurisprudenciaCitada',
    'notas',
    'informacoesComplementares',
    'termosAuxiliares',
    'teseJuridica',
    'tema',
    'referenciasLegislativas',
    'acordaosSimilares',
]

# ─── Arquivos ─────────────────────────────────────────────────────────────────
PARQUET_BASE  = 'espelhos_acordaos_artigo2026.parquet'
PARQUET_SAIDA = 'espelhos_acordaos_artigo2026_com_texto.parquet'

# Pasta local para cache dos ZIPs e JSONs (evita re-download)
DOWNLOAD_DIR = Path('downloads_stj')
DOWNLOAD_DIR.mkdir(exist_ok=True)
ESPELHOS_DIR = Path('downloads_stj/espelhos')
ESPELHOS_DIR.mkdir(exist_ok=True)

# Define se tenta baixar os zips das íntegras e dos espelhos ou apenas consolida o que já existir
CKAN_TIMEOUT = 600
BAIXAR_ESPELHOS_CKAN = True
BAIXAR_INTEGRAS_CKAN = True

# Coluna chave de junção entre parquet e ZIPs de íntegras
COL_SEQ = 'seq_documento_acordao'
# Coluna chave de junção entre parquet e JSONs de espelhos
COL_REG = 'num_registro'

print(f'CKAN        : {CKAN_BASE_URL}')
print(f'Dataset txt : {DATASET_TEXTOS}')
print(f'Parquet     : {PARQUET_BASE}')
print(f'Cache dir   : {DOWNLOAD_DIR.resolve()}')
print(f'Espelhos dir: {ESPELHOS_DIR.resolve()}')

## 2. Carregamento do Parquet Base

In [ ]:
df = pd.read_parquet(PARQUET_BASE)

print(f'Total de registros : {len(df)}')
print(f'Colunas            : {df.columns.tolist()}')
print()
df.head(3)

In [ ]:
# ── Lookup: seq_documento_acordao → índices do dataframe (para junção de textos) ──
assert COL_SEQ in df.columns, f'❌ Coluna "{COL_SEQ}" não encontrada no parquet!'

seq_para_idx: dict[int, list[int]] = {}
for idx, val in df[COL_SEQ].dropna().items():
    seq = int(val)
    seq_para_idx.setdefault(seq, []).append(idx)

seqs_necessarios: set[int] = set(seq_para_idx.keys())

print(f'seq_documento_acordao únicos necessários: {len(seqs_necessarios)}')
print(f'Exemplos: {list(seqs_necessarios)[:5]}')

# ── Lookup: num_registro → seq_documento_acordao (para junção dos espelhos) ──
if COL_REG in df.columns:
    reg_para_seq: dict[str, int] = {
        str(row[COL_REG]).strip(): int(row[COL_SEQ])
        for _, row in df[[COL_REG, COL_SEQ]].dropna().iterrows()
    }
    registros_necessarios: set[str] = set(reg_para_seq.keys())
    print(f'\nnum_registro únicos necessários: {len(registros_necessarios)}')
    print(f'Exemplos: {list(registros_necessarios)[:5]}')
else:
    reg_para_seq = {}
    registros_necessarios = set()
    print(f'⚠️  Coluna "{COL_REG}" não encontrada — extração de espelhos desabilitada.')

# ── Anos presentes no parquet — filtro de ZIPs por prefixo de nome ──
anos_necessarios: set[str] = set()
if 'ano' in df.columns:
    anos_necessarios = {str(int(a)) for a in df['ano'].dropna().unique()}
print(f'\nAnos necessários: {sorted(anos_necessarios)}')

## 3. Obtenção dos Recursos CKAN

Uma única chamada de API por dataset lista todos os recursos disponíveis.
- **Seção 3a**: ZIPs de textos integrais
- **Seção 3b**: JSONs de espelhos por órgão julgador

In [ ]:
def obter_metadados_dataset(dataset_id: str) -> dict:
    """Retorna os metadados de um dataset do CKAN."""
    url = f'{CKAN_BASE_URL}/api/3/action/package_show'
    response = requests.get(url, params={'id': dataset_id}, timeout=30)
    response.raise_for_status()
    return response.json()['result']

print('✅ Função obter_metadados_dataset definida.')

### 3a. ZIPs de Textos Integrais

In [ ]:
print(f'Obtendo lista de recursos do dataset "{DATASET_TEXTOS}"...')
metadados = obter_metadados_dataset(DATASET_TEXTOS)

# Filtra apenas recursos ZIP
recursos_zip = [
    {'name': r['name'], 'url': r['url'], 'resource_id': r['id']}
    for r in metadados.get('resources', [])
    if r.get('name', '').lower().endswith('.zip') or r.get('format', '').upper() == 'ZIP'
]

print(f'Recursos ZIP encontrados: {len(recursos_zip)}')
for r in recursos_zip[:10]:
    print(f'  {r["name"]}')
if len(recursos_zip) > 10:
    print(f'  ... (+{len(recursos_zip)-10} mais)')

### 3b. JSONs de Espelhos de Acórdãos

Busca os recursos JSON de cada dataset de espelho por órgão julgador.
Filtra apenas os recursos cujo nome indica um ano presente no parquet, `{ANO}_{MES}.json`.

In [ ]:
# Coleta os recursos JSON de espelhos de todos os órgãos julgadores
# Retorna lista de dicts: {'orgao', 'name', 'url', 'resource_id'}
recursos_espelho: list[dict] = []

for orgao, dataset_id in tqdm(DATASETS_ESPELHOS, desc='Órgãos'):
    try:
        meta = obter_metadados_dataset(dataset_id)
        for r in meta.get('resources', []):
            nome = r.get('name', '')
            fmt  = r.get('format', '').upper()
            # Aceita recursos JSON pelo formato ou extensão do nome
            if not (nome.lower().endswith('.json') or fmt == 'JSON'):
                continue
            # Filtra por ano quando sabemos os anos necessários
            if anos_necessarios:
                # Nome do recurso costuma começar com AAAA ou AAAA_MM
                # Ex.: "2023_01.json", "2023.json"
                ano_recurso = nome[:4]  # primeiros 4 caracteres
                if ano_recurso not in anos_necessarios:
                    continue
            recursos_espelho.append({
                'orgao'      : orgao,
                'name'       : f'{orgao}_{nome}',   # prefixo com órgão para evitar colisão
                'url'        : r['url'],
                'resource_id': r['id'],
            })
    except Exception as e:
        print(f'  ⚠️  Erro ao obter metadados de {orgao}/{dataset_id}: {e}')

print(f'\nTotal de recursos JSON de espelhos encontrados: {len(recursos_espelho)}')
for r in recursos_espelho[:10]:
    print(f'  [{r["orgao"]}] {r["name"]}')
if len(recursos_espelho) > 10:
    print(f'  ... (+{len(recursos_espelho)-10} mais)')

## 4. Extração dos Textos Integrais dos ZIPs

Cada ZIP é baixado com cache local (`downloads_stj/`).
O conteúdo é varrido em memória — apenas os TXTs com
`seq_documento_acordao` necessários são lidos.

In [ ]:
def baixar_com_cache(url: str, nome_arquivo: str, pasta: Path = DOWNLOAD_DIR) -> Path:
    """Baixa o arquivo para `pasta` se não existir. Retorna o caminho local."""
    caminho = pasta / nome_arquivo
    if caminho.is_file():
        print(f'  [cache] {nome_arquivo:<45}', end='\r', flush=True)
        return caminho
    if not BAIXAR_INTEGRAS_CKAN:
        raise Exception(f' [ignorado] Arquivo {nome_arquivo} ignorado (BAIXAR_INTEGRAS_CKAN=false)')
    print(f'  [↓] {nome_arquivo:<45}', end='\r', flush=True)
    with requests.get(url, stream=True, timeout=CKAN_TIMEOUT) as r:
        r.raise_for_status()
        with open(caminho, 'wb') as f:
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
    print(f'  [✓] {nome_arquivo:<45}', end='\r', flush=True)
    return caminho


def processar_zip(
    caminho_zip: Path,
    seqs_validos: set[int],
    textos: dict[int, str],
) -> int:
    """
    Varre o ZIP em memória.
    Para cada TXT cujo nome (stem) é um seq_documento_acordao no conjunto,
    lê o conteúdo e armazena em `textos[seq]`.
    Retorna o número de textos extraídos neste ZIP.
    """
    extraidos = 0
    with zipfile.ZipFile(caminho_zip, 'r') as zf:
        for membro in zf.namelist():
            if not membro.lower().endswith('.txt'):
                continue
            stem = Path(membro).stem
            try:
                seq = int(stem)
            except ValueError:
                continue
            if seq not in seqs_validos:
                continue
            # Lê o conteúdo do TXT (tenta UTF-8, fallback latin-1)
            raw = zf.read(membro)
            try:
                txt = raw.decode('utf-8')
            except UnicodeDecodeError:
                txt = raw.decode('latin-1')
            txt = txt.replace('<br>', '\n').strip()
            textos[seq] = txt
            extraidos += 1
    return extraidos


print('✅ Funções de extração de textos definidas.')

In [ ]:
# Dicionário: seq_documento_acordao → texto
textos_extraidos: dict[int, str] = {}

# Conjunto ainda não encontrado (para parar cedo se já tivermos tudo)
seqs_pendentes = set(seqs_necessarios)

# Filtra apenas os ZIPs cujo nome começa com um dos anos de interesse
if anos_necessarios:
    recursos_filtrados = [
        r for r in recursos_zip
        if any(r['name'].startswith(ano) for ano in anos_necessarios)
    ]
else:
    recursos_filtrados = recursos_zip

print(f'Iniciando extração de {len(seqs_pendentes)} textos em {len(recursos_filtrados)} ZIPs'
      + (f' [anos: {sorted(anos_necessarios)}]' if anos_necessarios else ' [todos os anos]') + '...')
print('Caso ocorra erro em algum arquivo, pode-se tentar novamente. Os arquivos são baixados na pasta "./downloads_stj"')

for recurso in tqdm(recursos_filtrados, desc='ZIPs'):
    if not seqs_pendentes:
        print('\n✅ Todos os textos encontrados — interrompendo varredura antecipada.')
        break

    try:
        caminho = baixar_com_cache(recurso['url'], recurso['name'])
        n = processar_zip(caminho, seqs_pendentes, textos_extraidos)
        # Remove os encontrados do conjunto pendente
        seqs_pendentes -= set(textos_extraidos.keys())
        if n:
            print(f'  → {n} texto(s) extraído(s) | pendentes: {len(seqs_pendentes)}', end='\r', flush=True)
    except Exception as e:
        print(f'  ⚠️  Erro em {recurso["name"]}: {e}')

print()
print(f'Textos extraídos : {len(textos_extraidos)} / {len(seqs_necessarios)}')
print(f'Não encontrados  : {len(seqs_pendentes)}')
if seqs_pendentes:
    print(f'  Exemplos: {list(seqs_pendentes)[:5]}')

## 5. Extração dos Dados de Espelho dos JSONs

Cada JSON de espelho é baixado com cache local (`downloads_stj/espelhos/`).
O arquivo é processado em memória — apenas os registros cujo `numeroRegistro`
esteja no parquet base são extraídos, e correlacionados com `seq_documento_acordao`
usando o lookup construído na etapa 2.

In [ ]:
import re as _re

def _formatar_valor(valor):
    """Normaliza strings (remove quebras de linha) e arredonda floats."""
    if isinstance(valor, str):
        return valor.replace('\n', ' ').replace('\r', ' ').strip()
    if isinstance(valor, float):
        return round(valor, 3)
    return valor


def processar_json_espelho(
    caminho_json: Path,
    registros_necessarios: set[str],
    reg_para_seq: dict[str, int],
    espelhos: dict[int, dict],
) -> int:
    """
    Lê o JSON de espelhos e filtra apenas os registros cujo `numeroRegistro`
    esteja em `registros_necessarios`.

    Correlaciona o `numeroRegistro` com `seq_documento_acordao` via `reg_para_seq`
    e armazena os campos de espelho em `espelhos[seq]`.

    Retorna o número de registros extraídos deste JSON.
    """
    extraidos = 0
    try:
        with open(caminho_json, 'r', encoding='utf-8') as f:
            dados = json.load(f)
    except UnicodeDecodeError:
        with open(caminho_json, 'r', encoding='latin-1') as f:
            dados = json.load(f)
    except json.JSONDecodeError as e:
        # Arquivo pode conter mensagem de erro do CKAN (ex.: "sem lançamentos")
        conteudo = caminho_json.read_text(encoding='utf-8', errors='replace')
        if 'sem lançamentos' in conteudo.lower():
            return 0
        raise

    if not isinstance(dados, list):
        dados = [dados]  # alguns arquivos retornam objeto único

    for item in dados:
        num_registro = str(item.get('numeroRegistro', '') or '').strip()
        if not num_registro or num_registro not in registros_necessarios:
            continue

        seq = reg_para_seq.get(num_registro)
        if seq is None:
            continue

        # Já temos um espelho para este seq — mantém o primeiro encontrado
        if seq in espelhos:
            continue

        registro = {c: _formatar_valor(item.get(c)) for c in COLUNAS_ESPELHO}
        registro['id_espelho']           = item.get('id')
        registro['seq_documento_acordao'] = seq
        registro['num_registro']          = num_registro
        espelhos[seq] = registro
        extraidos += 1

    return extraidos


print('✅ Funções de extração de espelhos definidas.')

In [ ]:
# Dicionário: seq_documento_acordao → dict com campos do espelho
espelhos_extraidos: dict[int, dict] = {}

if not registros_necessarios:
    print('⚠️  Nenhum num_registro disponível — extração de espelhos ignorada.')
elif not recursos_espelho:
    print('⚠️  Nenhum recurso JSON de espelho encontrado no CKAN.')
else:
    seqs_espelho_pendentes = set(reg_para_seq.values())  # seqs ainda sem espelho

    print(f'Iniciando extração de espelhos para {len(registros_necessarios)} registros'
          + (f' [anos: {sorted(anos_necessarios)}]' if anos_necessarios else '') + '...')
    print(f'Total de JSONs de espelhos a processar: {len(recursos_espelho)}')
    print('Os arquivos são baixados com cache em "./downloads_stj/espelhos/"')

    for recurso in tqdm(recursos_espelho, desc='Espelhos'):
        if not seqs_espelho_pendentes:
            print('\n✅ Todos os espelhos encontrados — interrompendo varredura antecipada.')
            break

        try:
            caminho = baixar_com_cache(recurso['url'], recurso['name'], pasta=ESPELHOS_DIR)
            n = processar_json_espelho(
                caminho,
                registros_necessarios,
                reg_para_seq,
                espelhos_extraidos,
            )
            # Remove os encontrados do conjunto pendente
            seqs_espelho_pendentes -= set(espelhos_extraidos.keys())
            if n:
                print(
                    f'  [{recurso["orgao"]}] {recurso["name"]}: {n} espelho(s) | '
                    f'pendentes: {len(seqs_espelho_pendentes)}',
                    end='\r', flush=True,
                )
        except Exception as e:
            print(f'  ⚠️  Erro em {recurso["name"]}: {e}')

    print()
    print(f'Espelhos extraídos : {len(espelhos_extraidos)} / {len(registros_necessarios)}')
    print(f'Não encontrados    : {len(seqs_espelho_pendentes)}')

## 6. Junção dos Textos e Espelhos ao Dataframe

In [ ]:
df_resultado = df.copy()

# ── Coluna 'texto' via seq_documento_acordao ──────────────────────────────────
df_resultado['texto'] = df_resultado[COL_SEQ].apply(
    lambda seq: textos_extraidos.get(int(seq), '') if pd.notna(seq) else ''
)

# ── Colunas do espelho via seq_documento_acordao ──────────────────────────────
if espelhos_extraidos:
    # Constrói um DataFrame com os dados de espelho indexado por seq_documento_acordao
    df_espelhos = pd.DataFrame(espelhos_extraidos.values()).set_index('seq_documento_acordao')
    # Remove colunas que já existem no df_resultado para evitar duplicatas
    colunas_novas = [c for c in df_espelhos.columns if c not in df_resultado.columns]
    df_espelhos = df_espelhos[colunas_novas]
    df_resultado = df_resultado.join(df_espelhos, on=COL_SEQ, how='left')
    print(f'Colunas do espelho adicionadas: {colunas_novas}')

# ── Estatísticas ──────────────────────────────────────────────────────────────
com_texto  = df_resultado['texto'].str.len().gt(0).sum()
sem_texto  = len(df_resultado) - com_texto
com_espelho = df_resultado['id_espelho'].notna().sum() if 'id_espelho' in df_resultado.columns else 0

print()
print('=== RESULTADO ===')
print(f'Total de registros : {len(df_resultado)}')
print(f'Com texto          : {com_texto}')
print(f'Sem texto          : {sem_texto}')
print(f'Com espelho        : {com_espelho}')
print(f'Colunas finais     : {df_resultado.columns.tolist()}')
if com_texto > 0:
    print('\nTamanho médio do texto (chars): ',
          df_resultado.loc[df_resultado['texto'].str.len()>0, 'texto'].str.len().mean().round(0))

## 7. Preview

In [ ]:
# Exibe exemplos de registros com texto e campos de espelho extraídos
com_texto_df = df_resultado[df_resultado['texto'].str.len() > 0]

if not com_texto_df.empty:
    for i, row in com_texto_df[:2].iterrows():
        dados = [f'{str(c).ljust(25)}: {v}' for c, v in row.items() if c != 'texto' and v]
        dados.sort()
        [print(_) for _ in dados]
        _texto = f'{row["texto"][:100]}\n[..]{row["texto"][-100:]}'.replace('\r','')
        print(f'TEXTO: {_texto}')
        print('-' * 60)
else:
    print('⚠️  Nenhum texto encontrado ainda. Verifique os ZIPs disponíveis no CKAN.')

## 8. Salvamento do Resultado

In [ ]:
df_resultado.to_parquet(PARQUET_SAIDA, index=False)

# ── Métricas básicas ──────────────────────────────────────────────────────────
total        = len(df_resultado)
com_texto    = df_resultado['texto'].str.len().gt(0).sum()
sem_texto    = total - com_texto
com_espelho  = df_resultado['id_espelho'].notna().sum() if 'id_espelho' in df_resultado.columns else 0
sem_espelho  = total - com_espelho

tamanhos     = df_resultado.loc[df_resultado['texto'].str.len() > 0, 'texto'].str.len()
txt_media    = tamanhos.mean()    if com_texto else 0
txt_mediana  = tamanhos.median()  if com_texto else 0
txt_min      = tamanhos.min()     if com_texto else 0
txt_max      = tamanhos.max()     if com_texto else 0

sep = '─' * 55
print(sep)
print('  📄  ARQUIVO GERADO')
print(sep)
print(f'  Arquivo   : {PARQUET_SAIDA}')
print(f'  Tamanho   : {Path(PARQUET_SAIDA).stat().st_size / 1024**2:.2f} MB')
print(f'  Colunas   : {len(df_resultado.columns)}  →  {df_resultado.columns.tolist()}')

print(sep)
print('  📊  COBERTURA DOS DADOS')
print(sep)
print(f'  Total de registros        : {total:>6}')
print(f'  Com texto integral        : {com_texto:>6}  ({com_texto/total*100:.1f}%)')
print(f'  Sem texto integral        : {sem_texto:>6}  ({sem_texto/total*100:.1f}%)')
print(f'  Com dados de espelho      : {com_espelho:>6}  ({com_espelho/total*100:.1f}%)')
print(f'  Sem dados de espelho      : {sem_espelho:>6}  ({sem_espelho/total*100:.1f}%)')

print(sep)
print('  📏  TAMANHO DOS TEXTOS INTEGRAIS (chars)')
print(sep)
if com_texto > 0:
    print(f'  Média                     : {txt_media:>10.0f}')
    print(f'  Mediana                   : {txt_mediana:>10.0f}')
    print(f'  Mínimo                    : {txt_min:>10.0f}')
    print(f'  Máximo                    : {txt_max:>10.0f}')
    print(f'  Total de caracteres       : {tamanhos.sum():>10,.0f}')
else:
    print('  (nenhum texto disponível)')

if 'ano' in df_resultado.columns:
    print(sep)
    print('  📅  REGISTROS POR ANO')
    print(sep)
    por_ano = df_resultado.groupby('ano').agg(
        total     = ('texto', 'count'),
        com_texto = ('texto', lambda x: x.str.len().gt(0).sum()),
    ).sort_index()
    for ano, row in por_ano.iterrows():
        pct = row['com_texto'] / row['total'] * 100 if row['total'] else 0
        print(f'  {int(ano)}  →  {row["total"]:>5} registros | texto: {row["com_texto"]:>5} ({pct:.0f}%)')

if 'nomeOrgaoJulgador' in df_resultado.columns:
    print(sep)
    print('  ⚖️   REGISTROS POR ÓRGÃO JULGADOR (top 10)')
    print(sep)
    por_orgao = (
        df_resultado['nomeOrgaoJulgador']
        .fillna('(não informado)')
        .value_counts()
        .head(10)
    )
    for orgao, cnt in por_orgao.items():
        print(f'  {str(orgao):<35} : {cnt:>5}')

campos_espelho_disponiveis = [c for c in COLUNAS_ESPELHO if c in df_resultado.columns]
if campos_espelho_disponiveis:
    print(sep)
    print('  📋  PREENCHIMENTO DOS CAMPOS DO ESPELHO')
    print(sep)
    for campo in campos_espelho_disponiveis:
        if df_resultado[campo].dtype == object:
            preenchidos = df_resultado[campo].fillna('').str.strip().ne('').sum()
        else:
            preenchidos = df_resultado[campo].notna().sum()
        pct = preenchidos / total * 100
        print(f'  {campo:<30} : {preenchidos:>5}  ({pct:.0f}%)')

print(sep)
print('  ✅  Processamento concluído!')
print(sep)
